# topology_workflow usage

This notebook records the stable module entry points migrated from `paper1notebook/codex2`.

The examples are intentionally small.  Use the paper1 pipeline scripts for full G60 runs.

## 1. Dynamic schedule from a comparison CSV

Input requirement: a CSV with a `step` column and topology columns such as `motif_000001`, `motif_000002`, ...

In [ ]:
import sys
from pathlib import Path

GENERIC_ROOT = Path(r"E:\\paper11\\generic")
if str(GENERIC_ROOT) not in sys.path:
    sys.path.insert(0, str(GENERIC_ROOT))

from src.topology_workflow.module import write_dynamic_schedule_outputs

WORK_DIR = Path(r"E:\\paper11\\data\\satnet_experiments\\_notebook_demo\\dynamic_schedule")
WORK_DIR.mkdir(parents=True, exist_ok=True)
COMPARE_CSV = WORK_DIR / "compare.csv"
COMPARE_CSV.write_text(
    "step,motif_000001,motif_000002,full_link\n"
    "0,3.0,4.0,2.5\n"
    "60,3.2,2.9,2.4\n"
    "120,4.1,2.7,2.6\n"
    "180,2.8,3.5,2.5\n",
    encoding="utf-8",
)

meta = write_dynamic_schedule_outputs(
    compare_csv=COMPARE_CSV,
    out_dir=WORK_DIR / "out",
    topology_prefix="motif_",
    min_dwell_minutes=[1, 2],
    metric_name="mean shortest delay (ms)",
)
meta

## 2. Paper1 shortest-delay pipeline command

For real G60 motif experiments, call the paper1 pipeline.  The module code remains parameter-transparent; concrete G60 paths live in YAML.

In [ ]:
from pathlib import Path

PYTHON = Path(r"C:\\ProgramData\\miniconda3\\envs\\paper11\\python.exe")
SCRIPT = Path(r"E:\\paper11\\generic\\paper1notebook\\pipeline\\run_paper1_motif_shortest_delay.py")
CONFIG = Path(r"E:\\paper11\\generic\\paper1notebook\\pipeline\\configs\\g60_w_le4_h_le3_shortest_delay.yaml")

cmd = [
    str(PYTHON),
    str(SCRIPT),
    "--config", str(CONFIG),
    "--end", "120",
    "--stride", "60",
    "--limit-motifs", "2",
    "--pairs", "china_europe",
    "--skip-gridplus",
]
cmd

## 3. Weighted edge betweenness core

`weighted_edge_betweenness_between_node_sets` only needs an edge table, a weight vector, and two node sets.  The weight vector can come from delay, bandwidth cost, or any other edge metric.

In [ ]:
import sys
from pathlib import Path
import numpy as np

GENERIC_ROOT = Path(r"E:\\paper11\\generic")
if str(GENERIC_ROOT) not in sys.path:
    sys.path.insert(0, str(GENERIC_ROOT))

from src.link_delay.module.edge_options import EdgeTable
from src.topology_metrics.module import weighted_edge_betweenness_between_node_sets

edge_table = EdgeTable(
    src=np.array([0, 1, 0], dtype=np.int32),
    dst=np.array([1, 2, 2], dtype=np.int32),
    option=np.array([0, 0, 0], dtype=np.int16),
    src_plane=np.zeros(3, dtype=np.int16),
    src_y=np.zeros(3, dtype=np.int16),
    dst_plane=np.zeros(3, dtype=np.int16),
    dst_y=np.zeros(3, dtype=np.int16),
    sat_ids=["0", "1", "2"],
)

values, summary, samples = weighted_edge_betweenness_between_node_sets(
    edge_table,
    total_nodes=3,
    source_nodes=[0],
    target_nodes=[2],
    weights=np.array([1.0, 1.0, 3.0], dtype=np.float32),
    sample_path_limit=1,
)
values, summary, samples